In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Cek Kelengkapan README.md Final & Ekstrak Data Aktual untuk Seluruh Dokumen**

In [6]:
import os
import pandas as pd

os.chdir("/content/drive/MyDrive/magang-ba-2026")

print("=== 1. SKEMA EKSAK 4 TABEL DARI src/db.py ===")
with open("src/db.py", "r", encoding="utf-8") as f:
    print(f.read())

print("\n=== 2. ISI src/export.py ===")
if os.path.exists("src/export.py"):
    with open("src/export.py", "r", encoding="utf-8") as f:
        print(f.read())

print("\n=== 3. STATISTIK AKTUAL comments_v1.csv ===")
df_v1 = pd.read_csv("data/processed/comments_v1.csv")
print("Kolom comments_v1.csv:", list(df_v1.columns))
print("Total baris:", len(df_v1), "| Unik comment_id:", df_v1["comment_id"].nunique())
for col in df_v1.columns:
    if any(k in col.lower() for k in ["cat", "kateg", "label", "plat", "split", "tgl", "date", "time"]):
        if df_v1[col].nunique() <= 20:
            print(f"\nSebaran '{col}':")
            print(df_v1[col].value_counts(dropna=False).to_string())
        else:
            print(f"\nRentang '{col}': {df_v1[col].dropna().min()} s.d. {df_v1[col].dropna().max()}")

print("\n=== 4. DAFTAR DOKUMEN DI docs/ DAN reports/ ===")
for folder in ["docs", "reports"]:
    if os.path.exists(folder):
        for fn in sorted(os.listdir(folder)):
            print(f" - {folder}/{fn}")

=== 1. SKEMA EKSAK 4 TABEL DARI src/db.py ===
import os
import sqlite3
import csv
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DB_PATH = str(PROJECT_ROOT / "data" / "app.db")

def get_conn():
    """Membuat koneksi SQLite dengan mengaktifkan foreign_keys = ON"""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH, check_same_thread=False, timeout=30)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def init_db():
    """Menginisialisasi 4 tabel utama pada database SQLite"""
    conn = get_conn()
    cursor = conn.cursor()

    # 1. Tabel posts (Unggahan)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS posts (
        post_id TEXT PRIMARY KEY,
        platform TEXT,
        published_at TEXT,
        text_content TEXT
    )
    """)

    # 2. Tabel comments (Komentar)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS comments (
        comment_id TEXT PRIMARY KEY,
        post_id TEXT,
   

**Mengisi data tictionary**

In [9]:
import os

os.chdir("/content/drive/MyDrive/magang-ba-2026")
os.makedirs("docs", exist_ok=True)

data_dict_content = """# Data Dictionary Final — Aplikasi Analisis Konten dan Sentimen Media Sosial

- **Versi Dokumen:** Final (`v1.0`)
- **Tanggal Pembaruan:** 27 November 2026
- **Basis Data Rujukan:** SQLite (`data/app.db`, skema didefinisikan pada `src/db.py`)
- **Kontrak Ekspor Rujukan:** `src/export.py` (`EXPORT_COLUMNS`, 21 kolom terstandar tanpa PII)

---

## Bagian 1: Skema Empat Tabel Utama Basis Data SQLite (`data/app.db`)

### 1.1 Tabel `posts` (Data Unggahan Induk)
Menyimpan metadata unggahan resmi dari akun media sosial Pemprov Jawa Timur.

| No | Nama Kolom | Tipe Data | Kunci / Batasan | Deskripsi | Contoh Nilai |
| :---: | :--- | :--- | :--- | :--- | :--- |
| 1 | `post_id` | `TEXT` | `PRIMARY KEY` | Pengenal unik unggahan lintas platform | `IG-P004` |
| 2 | `platform` | `TEXT` | Opsional | Nama platform media sosial sumber unggahan | `Instagram`, `TikTok` |
| 3 | `published_at` | `TEXT` | Opsional | Tanggal publikasi unggahan (format ISO / `DD-MM-YYYY`) | `2026-09-29` |
| 4 | `text_content` | `TEXT` | Opsional | Teks takarir (*caption*) atau deskripsi unggahan | `Capaian pembangunan infrastruktur Jatim...` |

---

### 1.2 Tabel `comments` (Data Komentar Pengguna)
Menyimpan teks komentar yang telah dibersihkan, dianonimisasi, dan dipetakan ke unggahan induknya.

| No | Nama Kolom | Tipe Data | Kunci / Batasan | Deskripsi | Contoh Nilai |
| :---: | :--- | :--- | :--- | :--- | :--- |
| 1 | `comment_id` | `TEXT` | `PRIMARY KEY` | Pengenal unik komentar pada dataset | `IG-P004-C001` |
| 2 | `post_id` | `TEXT` | `FOREIGN KEY` $\rightarrow$ `posts(post_id)` | Referensi pengenal unggahan induk | `IG-P004` |
| 3 | `text_content` | `TEXT` | Opsional | Teks komentar hasil normalisasi/pembersihan | `Bojonegoro lanjutkan!` |
| 4 | `text_original` | `TEXT` | Opsional | Teks komentar asli setelah anonimisasi `@user` | `Bojonegoro..... Lanjutkan!` |
| 5 | `text_hash` | `TEXT` | Opsional | Nilai *hash* teks untuk deteksi duplikasi cepat | `a94a8fe5ccb19ba6...` |
| 6 | `published_at` | `TEXT` | Opsional | Tanggal publikasi unggahan/komentar rujukan | `2026-09-29` |
| 7 | `commented_at` | `TEXT` | Opsional | Tanggal waktu komentar dikirimkan pengguna | `2026-09-30` |
| 8 | `context_text` | `TEXT` | Opsional | Ringkasan topik atau konteks unggahan induk | `Prestasi dan kebanggaan daerah kabupaten/kota...` |
| 9 | `content_category` | `TEXT` | Opsional | Kategori tematik konten unggahan | `Infrastruktur`, `Pendidikan`, `Pelayanan Publik` |
| 10 | `is_eligible` | `INTEGER` | `1` = Layak, `0` = Tidak | Penanda kelayakan komentar untuk analisis sentimen | `1` |
| 11 | `status_eksklusi` | `TEXT` | Baku: `'Aktif'` | Status keikutsertaan baris dalam dataset analisis | `Aktif`, `Dieksklusi` |
| 12 | `alasan_eksklusi` | `TEXT` | Baku: `'Lolos'` | Keterangan alasan apabila komentar tidak diikutsertakan | `Lolos`, `Q03: ID duplikat` |
| 13 | `split` | `TEXT` | Baku: `'train'` | Pembagian himpunan data eksperimen model | `train`, `validation`, `test` |

---

### 1.3 Tabel `predictions` (Hasil Prediksi Model Sentimen)
Menyimpan keluaran inferensi otomatis dari model klasifikasi sentimen IndoBERT.

| No | Nama Kolom | Tipe Data | Kunci / Batasan | Deskripsi | Contoh Nilai |
| :---: | :--- | :--- | :--- | :--- | :--- |
| 1 | `prediction_id` | `INTEGER` | `PRIMARY KEY AUTOINCREMENT` | Pengenal unik catatan prediksi | `1` |
| 2 | `comment_id` | `TEXT` | `NOT NULL`, `FK` $\rightarrow$ `comments(comment_id)` | Pengenal komentar yang diprediksi | `IG-P004-C001` |
| 3 | `model_name` | `TEXT` | Opsional | Nama repositori/arsitektur model yang digunakan | `crypter70/IndoBERT-Sentiment-Analysis` |
| 4 | `model_version` | `TEXT` | Opsional | ID eksperimen atau versi konfigurasi model | `T3_indobert_len64` |
| 5 | `predicted_label` | `TEXT` | Domain: `positif`, `netral`, `negatif` | Kelas sentimen hasil pemetaan prediksi model | `positif` |
| 6 | `score` | `REAL` | Rentang `0.0` – `1.0` | Skor probabilitas keyakinan kelas terpilih | `0.9015` |
| 7 | `score_type` | `TEXT` | Opsional | Jenis fungsi skor probabilitas model | `softmax` |
| 8 | `predicted_at` | `TEXT` | Format ISO-8601 | Stempel waktu saat prediksi dijalankan | `2026-11-23T13:52:22` |
| 9 | `run_id` | `TEXT` | Opsional | Pengenal sesi eksekusi inferensi *batch* | `run-20261123-135222` |

---

### 1.4 Tabel `verification_history` (Riwayat Koreksi & Audit Verifikasi)
Mencatat setiap perubahan label sentimen yang dilakukan secara manual oleh peneliti/verifikator.

| No | Nama Kolom | Tipe Data | Kunci / Batasan | Deskripsi | Contoh Nilai |
| :---: | :--- | :--- | :--- | :--- | :--- |
| 1 | `history_id` | `INTEGER` | `PRIMARY KEY AUTOINCREMENT` | Pengenal unik log riwayat verifikasi | `1` |
| 2 | `comment_id` | `TEXT` | `NOT NULL`, `FK` $\rightarrow$ `comments(comment_id)` | Pengenal komentar yang diverifikasi | `IG-P004-C004` |
| 3 | `prediction_id` | `INTEGER` | `FK` $\rightarrow$ `predictions(prediction_id)` | Referensi baris prediksi model sebelumnya | `4` |
| 4 | `previous_label` | `TEXT` | Opsional | Label sentimen sebelum dikoreksi | `netral` |
| 5 | `new_label` | `TEXT` | `NOT NULL` | Label sentimen baru hasil keputusan peninjau | `negatif` |
| 6 | `reason` | `TEXT` | Opsional | Alasan atau catatan pembenaran koreksi label | `Keluhan kondisi jalan berdebu jelas bernada negatif` |
| 7 | `reviewer_id` | `TEXT` | Opsional | Kode atau identitas peninjau/peneliti | `R1_Azzahra` |
| 8 | `verified_at` | `TEXT` | `NOT NULL` (ISO-8601) | Stempel waktu saat verifikasi disimpan | `2026-11-23T14:05:10` |

> **Catatan Tabel Pendukung:** Selain 4 tabel operasional di atas, `src/db.py` juga menyediakan tabel `annotators` (`annotator_id`, `name`) dan `annotations` (`annotation_id`, `comment_id`, `annotator_id`, `label`, `created_at`) untuk menyimpan arsip pelabelan manual awal.

---

## Bagian 2: Skema Kolom Ekspor CSV (`data.csv`) & Metadata Manifest

Setiap kali pengguna mengekspor data melalui halaman **6_Ekspor** (`src/export.py`), aplikasi menghasilkan berkas `data.csv` yang menggabungkan atribut unggahan, komentar, prediksi terbaru, dan hasil verifikasi dalam **21 kolom terstandar**. Kolom identitas pengguna (`username`, `user_name`, `author`, `user_id`) **dihapus otomatis** sebelum ekspor demi perlindungan privasi (PII).

| No | Nama Kolom Ekspor | Sumber / Asal Data | Tipe | Deskripsi |
| :---: | :--- | :--- | :--- | :--- |
| 1 | `comment_id` | `comments.comment_id` | `TEXT` | Pengenal unik komentar |
| 2 | `post_id` | `comments.post_id` | `TEXT` | Pengenal unik unggahan induk |
| 3 | `platform` | `posts.platform` | `TEXT` | Platform sumber (`Instagram` / `TikTok`) |
| 4 | `source_account` | Metadata unggahan | `TEXT` | Nama akun resmi instansi sumber (tanpa PII warga) |
| 5 | `post_url` | Metadata unggahan | `TEXT` | Tautan URL publik unggahan induk |
| 6 | `published_at` | `posts.published_at` | `TEXT` | Tanggal publikasi unggahan |
| 7 | `context_text` | `comments.context_text` | `TEXT` | Ringkasan konteks topik unggahan |
| 8 | `content_format` | Metadata unggahan | `TEXT` | Format media unggahan (foto, karosel, video/reels) |
| 9 | `content_category` | `comments.content_category` | `TEXT` | Kategori tematik konten unggahan |
| 10 | `commented_at` | `comments.commented_at` | `TEXT` | Tanggal komentar dikirimkan |
| 11 | `text_original` | `comments.text_original` | `TEXT` | Teks komentar yang sudah dianonimisasi |
| 12 | `split` | `comments.split` | `TEXT` | Pembagian himpunan data (`train` / `validation` / `test`) |
| 13 | `data_version` | Metadata dataset | `TEXT` | Versi paket dataset rujukan (contoh: `v1`) |
| 14 | `pred_label` | `predictions.predicted_label` | `TEXT` | Label sentimen prediksi model (`positif`, `netral`, `negatif`) |
| 15 | `pred_score` | `predictions.score` | `REAL` | Skor keyakinan prediksi model (`0.0` – `1.0`) |
| 16 | `score_type` | `predictions.score_type` | `TEXT` | Jenis skor probabilitas (`softmax`) |
| 17 | `model_version` | `predictions.model_version` | `TEXT` | Versi/ID eksperimen model (`T3_indobert_len64`) |
| 18 | `verified_label` | `verification_history.new_label` | `TEXT` | Label hasil verifikasi manual (kosong jika belum dikoreksi) |
| 19 | `verify_reason` | `verification_history.reason` | `TEXT` | Catatan alasan koreksi oleh verifikator |
| 20 | `reviewer_id` | `verification_history.reviewer_id` | `TEXT` | Identitas peninjau yang melakukan verifikasi |
| 21 | `verified_at` | `verification_history.verified_at` | `TEXT` | Stempel waktu tindakan verifikasi |

### Metadata Pendamping (`export_manifest.json`)
Bersamaan dengan `data.csv`, paket ZIP menyertakan `export_manifest.json` yang mencatat parameter keterlacakan (*reproducibility*): `exported_at`, `filters`, `n_rows`, `data_version`, `split_manifest`, `model_name`, `model_version`, `score_type`, `app_version`, dan daftar `columns`.
"""

with open("docs/data_dictionary_final.md", "w", encoding="utf-8") as f:
    f.write(data_dict_content)

print(f"File docs/data_dictionary_final.md berhasil dibuat ({os.path.getsize('docs/data_dictionary_final.md')} bytes).")

File docs/data_dictionary_final.md berhasil dibuat (8803 bytes).


**Menyusun docs/pedoman_label_final.md (memuat versi final, aturan tiap label, contoh ilustrasi aktual, riwayat perubahan, dan statistik kesepakatan aktual antar-anotator)**

In [10]:
import os

os.chdir("/content/drive/MyDrive/magang-ba-2026")

for rf in ["docs/pedoman_label_v0.1.md", "docs/D15_ringkasan_label.md", "docs/D29_petunjuk_aplikasi.md"]:
    if os.path.exists(rf):
        print(f"\n{'='*60}\nISI DOKUMEN REFERENSI: {rf}\n{'='*60}")
        with open(rf, "r", encoding="utf-8") as f:
            print(f.read())


ISI DOKUMEN REFERENSI: docs/pedoman_label_v0.1.md
# Pedoman Pelabelan Data Sentimen (Versi 0.1)

Dokumen ini menjadi acuan utama untuk melabeli sentimen dari komentar masyarakat pada media sosial Pemerintah Provinsi Jawa Timur.

### A. Kelas Sentimen Utama (Berdasarkan Bab 4.10)
1.  **Positif:** Komentar yang mengandung unsur apresiasi, dukungan, kepuasan, rasa syukur, atau kebanggaan terhadap kinerja/program Pemprov. 
    *   *Contoh:* "Mantap🔥", "Terbaik pak langkah yang sangat tepat"
2.  **Negatif:** Komentar yang mengandung unsur keluhan, kemarahan, kritik, kekecewaan, atau penolakan. 
    *   *Contoh:* "Gubernur Gak pecus blasss", "Roda 4 udah bertahun2 kagak ada pemutihan"
3.  **Netral:** Komentar yang bersifat informatif, tidak mengandung emosi yang jelas, atau sekadar memberikan fakta.
    *   *Contoh:* "Samsat Ngawi igne nopo nggeh?"

### B. Aturan Tambahan dari Kasus Pilot
Berdasarkan uji coba data *pilot*, disepakati penambahan 5 aturan khusus untuk menangani komentar yang 

**Membuat file docs/pedoman_label_final.md, dan docs/petunjuk_aplikasi_final.md**

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score

os.chdir("/content/drive/MyDrive/magang-ba-2026")
os.makedirs("docs", exist_ok=True)

# 1. Hitung statistik kesepakatan aktual dari data/processed/adjudication.csv
adj_path = "data/processed/adjudication.csv"
kappa_val, agree_pct, n_adj = "N/A", "N/A", 0
diff_count = 0

if os.path.exists(adj_path):
    df_adj = pd.read_csv(adj_path)
    print("Kolom adjudication.csv:", list(df_adj.columns), "| Jumlah baris:", len(df_adj))
    label_cols = [
        c for c in df_adj.columns
        if "label" in c.lower() and "final" not in c.lower() and "adj" not in c.lower()
    ]
    if len(label_cols) >= 2:
        c1, c2 = label_cols[0], label_cols[1]
        valid = df_adj[[c1, c2]].dropna()
        y1 = valid[c1].astype(str).str.strip().str.lower()
        y2 = valid[c2].astype(str).str.strip().str.lower()
        n_adj = len(valid)
        agree_pct = f"{accuracy_score(y1, y2) * 100:.2f}%"
        kappa_val = f"{cohen_kappa_score(y1, y2):.4f}"
        diff_count = int((y1 != y2).sum())
        print(f"Statistik Kesepakatan ({c1} vs {c2}): n={n_adj}, Agree={agree_pct}, Kappa={kappa_val}, Selisih={diff_count}")

# 2. Tulis docs/pedoman_label_final.md
pedoman_final = f"""# Pedoman Pelabelan Sentimen (Versi Final — v1.0)

- **Versi Dokumen:** Final (`v1.0`)
- **Tanggal Penetapan:** 27 November 2026
- **Penyusun:** Azzahra Attaqina (Laboratorium Business Analytics, Polinema)
- **Dataset Acuan:** Komentar media sosial resmi Pemerintah Provinsi Jawa Timur (`comments_v1.csv`, 1.000 baris)

---

## 1. Definisi Kelas Sentimen Utama

Setiap komentar masyarakat yang lolos penyaringan kelayakan diklasifikasikan ke dalam **3 kelas sentimen utama**:

| Kelas Label | Kode Sistem | Definisi Operasional | Contoh Ilustrasi Aktual |
| :--- | :---: | :--- | :--- |
| **Positif** | `positif` | Komentar yang menyampaikan apresiasi, dukungan, rasa syukur, pujian, kepuasan, atau kebanggaan terhadap program, kebijakan, maupun kinerja Pemprov Jawa Timur. | *"Mantap🔥"*, *"Terbaik pak langkah yang sangat tepat"*, *"Alhamdulillah masuk 10 besar"*, *"Bojonegoro..... Lanjutkan!"* |
| **Negatif** | `negatif` | Komentar yang mengandung keluhan, kemarahan, kritik keras, kekecewaan, protes, penolakan, atau sindiran tajam terhadap layanan/kebijakan. | *"Gubernur Gak pecus blasss"*, *"Roda 4 udah bertahun2 kagak ada pemutihan"*, *"Nganjuk opo2 kok Gleput 😜"* |
| **Netral** | `netral` | Komentar yang bersifat informatif, berupa pertanyaan prosedural tanpa emosi, sekadar menandai akun lain, atau meluruskan fakta secara objektif. | *"Samsat Ngawi igne nopo nggeh?"*, *"Roda 4 kapan ibu?"*, *"ini videonya bukan tembakau campalok"* |

---

## 2. Aturan Khusus Kasus Sulit & Eksklusi

Untuk menjaga konsistensi antar-anotator, ditetapkan **6 aturan operasional** berikut:

1. **Spam, Iklan & Bot (`SPAM` $\\rightarrow$ Eksklusi):** Komentar berisi promosi produk, tautan penipuan/judi, atau komentar otomatis tidak relevan tidak dilabeli sentimen dan diberi status **Eksklusi** (`is_eligible = 0`).
2. **Pertanyaan Layanan (`SULIT-TANYA` $\\rightarrow$ `netral`):** Pertanyaan murni mengenai jadwal, syarat, atau prosedur layanan publik tanpa nada marah/mengeluh wajib dilabeli **Netral**.
3. **Sarkasme & Sindiran (`SULIT-SARKASME` $\\rightarrow$ `negatif`):** Komentar yang memakai kata pujian atau emoji tertawa namun konteks kalimatnya menyindir/mengejek kebijakan (contoh: *"Syarat dan ketentuan Berlaku😂 yang desil 6-10 sini kumpul"*) wajib dilabeli **Negatif**.
4. **Koreksi Fakta (`SULIT-KOREKSI` $\\rightarrow$ `netral`):** Komentar yang meluruskan data atau visual unggahan tanpa ekspresi kemarahan/pujian dilabeli **Netral**.
5. **Bahasa Daerah (`SULIT-BAHASA` $\\rightarrow$ Tetap Layak):** Komentar berbahasa Jawa atau Madura tetap dinilai layak (`is_eligible = 1`); makna sentimen ditentukan berdasarkan arti kontekstualnya dan diberi catatan `RAGU` bila ambigu.
6. **Emotikon Tunggal & Tag Akun:** Komentar yang hanya berisi emotikon dukungan (mis. `👏👏👏`, `🔥`) dilabeli **Positif**, sedangkan komentar yang sekadar menyebut/menandai akun lain tanpa opini dilabeli **Netral** dengan catatan `RAGU - sekadar tag akun`.

---

## 3. Riwayat Perubahan Pedoman

| Versi | Tahap / Hari | Ringkasan Perubahan |
| :---: | :--- | :--- |
| `v0.1` | Hari 4–5 (Uji Pilot) | Penetapan 3 kelas sentimen dasar (`Positif`, `Negatif`, `Netral`) beserta 5 aturan penanganan kasus khusus (`SPAM`, `SULIT-TANYA`, `SULIT-SARKASME`, `SULIT-KOREKSI`, `SULIT-BAHASA`). |
| `v0.2` | Hari 11–15 (Pelabelan Batch 1–5) | Penambahan standarisasi kode catatan keraguan (`RAGU - sekadar tag akun`, `RAGU - sentimen ganda`, `teks asli cuma emoticon`) serta sesi pemeriksaan mandiri (*re-check*) yang mengoreksi **10 label** dari 1.000 komentar. |
| `v1.0` | Hari 16 & Hari 38 (Final) | Penguncian aturan *adjudication* bersama peninjau kedua (R2), pemetaan kode label huruf kecil (`positif`, `netral`, `negatif`) untuk integrasi model IndoBERT, serta dokumentasi statistik kesepakatan aktual. |

---

## 4. Statistik Kesepakatan & Sebaran Label Aktual

### A. Hasil Uji Kesepakatan Antar-Anotator (R1 vs R2)
Berdasarkan pemeriksaan silang buta (*blind review*) pada himpunan uji kesepakatan (`data/processed/adjudication.csv`):
- **Jumlah Sampel Uji Kesepakatan ($n$):** `{n_adj}` komentar
- **Tingkat Kesepakatan Mentah (*Raw Agreement*):** `{agree_pct}`
- **Koefisien *Cohen's Kappa* ($\\kappa$):** `{kappa_val}`
- **Jumlah Kasus Perbedaan yang Diadjudikasi:** `{diff_count}` komentar (diselesaikan melalui diskusi penentuan label acuan final).

### B. Rekapitulasi Sebaran Label Dataset `v1` (Batch 1–5)
- **Rekapitulasi Pemeriksaan Ulang (*Re-check* Mandiri):** **10 data** mengalami penyesuaian label selama proses kendali mutu batch 1–5.
- **Sebaran Label Sentimen Pelabelan Batch 1–5 (1.000 komentar):**
  - **Netral:** 802 komentar (80,2%)
  - **Positif:** 146 komentar (14,6%)
  - **Negatif:** 52 komentar (5,2%)
"""

with open("docs/pedoman_label_final.md", "w", encoding="utf-8") as f:
    f.write(pedoman_final)

# 3. Tulis docs/petunjuk_aplikasi_final.md
cb = chr(96) * 3
petunjuk_final = f"""# Petunjuk Penggunaan Aplikasi Final — Analisis Konten dan Sentimen Media Sosial

- **Versi Dokumen:** Final (`v1.0`)
- **Tanggal Rilis:** 27 November 2026
- **Penyusun:** Azzahra Attaqina (Laboratorium Business Analytics, Polinema)

---

## 1. Persiapan Lingkungan & Menjalankan Aplikasi

1. **Buka terminal** pada direktori proyek (`magang-ba-2026`) dan aktifkan lingkungan virtual:
   {cb}bash
   # Windows
   .venv\\Scripts\\activate

   # macOS / Linux
   source .venv/bin/activate
   {cb}

2. **Inisialisasi basis data SQLite** (hanya dilakukan saat pertama kali pemasangan atau saat mereset basis data):
   {cb}bash
   python -m src.db
   {cb}

3. **Jalankan antarmuka web Streamlit:**
   {cb}bash
   streamlit run app/Home.py
   {cb}

4. **Buka peramban** pada alamat `http://localhost:8501`.

---

## 2. Panduan Fitur per Halaman

### A. Halaman Utama (`Home`)
Menampilkan ringkasan tujuan aplikasi, status koneksi basis data SQLite (`data/app.db`), serta panduan singkat urutan alur kerja mulai dari impor data hingga ekspor laporan.

### B. Halaman Impor Data (`1_Impor`)
1. Pilih jenis data yang akan diunggah (**Posts** atau **Comments**).
2. Unggah **Posts** terlebih dahulu (mis. `posts_b1.csv`, `posts_b2.csv`, `posts_b3.csv`), periksa pratinjau 5 baris pertama, lalu klik **Mulai Impor Posts**.
3. Unggah **Comments** berikutnya (mis. `data/processed/comments_v1.csv`), tinjau laporan validasi kolom dan peringatan duplikasi (`Q03` ID duplikat / `Q08` teks duplikat), lalu klik **Mulai Impor Comments**.
4. Sistem secara idempotent akan menerima baris baru yang valid, melewati baris yang sudah tersimpan, dan mencatat alasan penolakan jika ada pelanggaran *foreign key* atau duplikasi.

### C. Halaman Eksplorasi & Filter Data (`2_Data`)
1. Gunakan panel **Filter Data Lanjutan** di bilah sisi kiri untuk menyaring komentar berdasarkan:
   - **Platform** (Instagram, TikTok, atau semua)
   - **Rentang Tanggal** (Dari Tanggal – Sampai Tanggal)
   - **Kategori Konten** (Infrastruktur, Pendidikan, Kesehatan, Pelayanan Publik, Transportasi, dll.)
   - **Kata Kunci Teks**
2. Tabel utama di sisi kanan menampilkan daftar komentar yang memenuhi kriteria secara *real-time*.

### D. Halaman Analisis Sentimen (`3_Analisis`)
1. Pastikan informasi model aktif menampilkan `crypter70/IndoBERT-Sentiment-Analysis` (`T3_indobert_len64`, `max_length: 64`, `score_type: softmax`).
2. Pilih cakupan sumber data komentar yang hendak dianalisis.
3. Klik tombol **Jalankan Analisis** dan tunggu bilah kemajuan hingga selesai.
4. Hasil prediksi (`positif`, `netral`, `negatif`), skor probabilitas, serta stempel waktu otomatis tersimpan ke tabel `predictions` dan ditampilkan pada pratinjau 10 riwayat prediksi terbaru.

### E. Halaman Verifikasi & Detail Prediksi (`4_Verifikasi`)
1. Tinjau tabel ringkasan hasil prediksi terbaru atau gunakan penyaring versi model di bilah kiri.
2. Pada **Panel Detail Komentar & Koreksi**, pilih `comment_id` untuk melihat teks komentar lengkap beserta konteks unggahan induknya.
3. Jika prediksi model perlu dikoreksi, pilih **Label Baru** (`positif`, `netral`, atau `negatif`), isi **ID Reviewer**, tuliskan **Alasan Koreksi**, lalu klik **Simpan Koreksi**.
4. Setiap koreksi tersimpan permanen sebagai jejak audit di tabel `verification_history`.

### F. Halaman Dashboard Ringkasan (`5_Dashboard`)
1. Gunakan panel **Filter Data Interaktif** di bilah sisi kiri (termasuk pilihan sumber label: *Prediksi model* atau *Label verifikasi*).
2. Amati kartu metrik KPI serta empat grafik utama: distribusi platform, distribusi kategori konten, distribusi sentimen, dan tren mingguan jumlah komentar.

### G. Halaman Ekspor Data (`6_Ekspor`)
1. Atur filter data yang ingin diunduh melalui panel samping.
2. Periksa pratinjau tabel ekspor (21 kolom terstandar tanpa identitas pribadi/PII).
3. Klik tombol unduh untuk memperoleh arsip ZIP yang berisi `data.csv` dan `export_manifest.json`.
"""

with open("docs/petunjuk_aplikasi_final.md", "w", encoding="utf-8") as f:
    f.write(petunjuk_final)

print("Langkah 3 selesai!")
print(f" - docs/pedoman_label_final.md   : {os.path.getsize('docs/pedoman_label_final.md')} bytes")
print(f" - docs/petunjuk_aplikasi_final.md: {os.path.getsize('docs/petunjuk_aplikasi_final.md')} bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kolom adjudication.csv: ['comment_id', 'konteks', 'teks', 'label_mhs', 'label_r2', 'label_final', 'alasan', 'adjudicator'] | Jumlah baris: 33
Statistik Kesepakatan (label_mhs vs label_r2): n=33, Agree=0.00%, Kappa=-0.2903, Selisih=33
Langkah 3 selesai!
 - docs/pedoman_label_final.md   : 4723 bytes
 - docs/petunjuk_aplikasi_final.md: 3918 bytes


In [12]:
import os
import json
import sqlite3
import pandas as pd

os.chdir("/content/drive/MyDrive/magang-ba-2026")

# 1. Cek output teks pada D16_adjudication_dan_svm_baseline.ipynb untuk mencari angka Kappa keseluruhan
nb_path = "docs/D16_adjudication_dan_svm_baseline.ipynb"
if os.path.exists(nb_path):
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    print("=== 1. OUTPUT DARI D16_adjudication_dan_svm_baseline.ipynb ===")
    for cell in nb.get("cells", []):
        for out in cell.get("outputs", []):
            txt = "".join(out.get("text", []))
            if any(k in txt.lower() for k in ["kappa", "agree", "sepakat", "adjudik", "total"]):
                print(txt.strip())
                print("-" * 40)

# 2. Cek sebaran aktual kategori konten & platform dari data/app.db dan file processed/
print("\n=== 2. STATISTIK AKTUAL KATEGORI & PLATFORM UNTUK 3 LEMBAR PELUANG ===")
if os.path.exists("data/app.db"):
    conn = sqlite3.connect("data/app.db")
    try:
        df_cat = pd.read_sql_query("""
            SELECT COALESCE(NULLIF(TRIM(content_category), ''), 'Tidak Berkategori') AS kategori,
                   COUNT(*) AS jumlah
            FROM comments
            GROUP BY kategori
            ORDER BY jumlah DESC
        """, conn)
        print("\nSebaran content_category di data/app.db:")
        print(df_cat.to_string(index=False))

        df_plat = pd.read_sql_query("""
            SELECT COALESCE(p.platform, 'Tidak Diketahui') AS platform,
                   COUNT(c.comment_id) AS jumlah_komentar
            FROM comments c
            LEFT JOIN posts p ON c.post_id = p.post_id
            GROUP BY platform
        """, conn)
        print("\nSebaran platform di data/app.db:")
        print(df_plat.to_string(index=False))
    except Exception as e:
        print("Info DB:", e)
    conn.close()

# Cek juga file posts & comments di data/processed/
for fn in sorted(os.listdir("data/processed")):
    if fn.endswith(".csv"):
        df_tmp = pd.read_csv(os.path.join("data/processed", fn))
        cat_cols = [c for c in df_tmp.columns if any(k in c.lower() for k in ["cat", "kateg", "tema", "topik"])]
        if cat_cols:
            print(f"\nFile data/processed/{fn} (baris={len(df_tmp)}), kolom kategori={cat_cols}:")
            print(df_tmp[cat_cols[0]].value_counts(dropna=False).to_string())

=== 1. OUTPUT DARI D16_adjudication_dan_svm_baseline.ipynb ===
Persentase Kesepakatan: 85.65%
----------------------------------------
Cohen's Kappa Score: 0.5390
----------------------------------------
Jumlah pasangan data yang dibandingkan: 230
Persentase Kesepakatan: 85.65%
Cohen's Kappa Score: 0.5390

=== TABEL SILANG (CROSS-TABULATION) ===
Penilai Kedua (R2)  Negatif  Netral  Positif
Mahasiswa                                   
Negatif                   1       0        0
Netral                   10     173       15
Positif                   3       5       23

----------------------------------------
Berhasil membuat file adjudikasi!
Total data berbeda label yang perlu direview: 33 baris
Disimpan di: data/processed/adjudication.csv
----------------------------------------
Total baris valid untuk pelatihan: 967 dari 1000 baris
----------------------------------------

=== 2. STATISTIK AKTUAL KATEGORI & PLATFORM UNTUK 3 LEMBAR PELUANG ===

Sebaran content_category di data/app.db:


In [13]:
import os
import pandas as pd

os.chdir("/content/drive/MyDrive/magang-ba-2026")

# 1. Perbarui Bagian 4 pada docs/pedoman_label_final.md dengan angka aktual D16
with open("docs/pedoman_label_final.md", "r", encoding="utf-8") as f:
    txt = f.read()

old_sec4 = txt[txt.index("## 4. Statistik Kesepakatan"):]
new_sec4 = """## 4. Statistik Kesepakatan & Sebaran Label Aktual

### A. Hasil Uji Kesepakatan Antar-Anotator (Mahasiswa R1 vs Penilai Kedua R2)
Berdasarkan pemeriksaan silang buta (*blind review*) pada himpunan uji kesepakatan (dokumentasi Hari 16):
- **Jumlah Pasangan Sampel Uji Kesepakatan ($n$):** **230 komentar**
- **Tingkat Kesepakatan Mentah (*Raw Agreement*):** **85,65%** (197 komentar sepakat langsung)
- **Koefisien *Cohen's Kappa* ($\kappa$):** **0,5390** (kategori *Moderate Agreement*)
- **Jumlah Kasus Perbedaan yang Diadjudikasi:** **33 baris** (`data/processed/adjudication.csv`), sehingga diperoleh **967 baris valid** untuk pelatihan setelah penyisihan kasus yang tetap ambigu.

| Penilai R1 \\ Penilai R2 | Negatif | Netral | Positif | Total R1 |
| :--- | :---: | :---: | :---: | :---: |
| **Negatif** | 1 | 0 | 0 | **1** |
| **Netral** | 10 | 173 | 15 | **198** |
| **Positif** | 3 | 5 | 23 | **31** |
| **Total R2** | **14** | **178** | **38** | **230** |

### B. Rekapitulasi Sebaran Label Dataset `v1` (Batch 1–5)
- **Rekapitulasi Pemeriksaan Ulang (*Re-check* Mandiri):** **10 data** mengalami penyesuaian label selama proses kendali mutu batch 1–5.
- **Sebaran Label Sentimen Pelabelan Batch 1–5 (1.000 komentar):**
  - **Netral:** 802 komentar (80,2%)
  - **Positif:** 146 komentar (14,6%)
  - **Negatif:** 52 komentar (5,2%)
"""

with open("docs/pedoman_label_final.md", "w", encoding="utf-8") as f:
    f.write(txt[:txt.index("## 4. Statistik Kesepakatan")] + new_sec4)

print("Bagian 4 docs/pedoman_label_final.md berhasil diperbarui dengan angka aktual D16!")

# 2. Cek distribusi kategori aktual dari posts_v0.csv + comments_v1.csv
df_posts = pd.read_csv("data/processed/posts_v0.csv")
df_comm = pd.read_csv("data/processed/comments_v1.csv")
print("\nKolom posts_v0.csv:", list(df_posts.columns))

# Gabungkan posts_v0 dan comments_v1 pada post_id untuk menghitung jumlah komentar aktual per kategori & platform
df_posts_unique = df_posts.drop_duplicates(subset=["post_id"])
df_merged = df_comm.merge(df_posts_unique, on="post_id", how="left", suffixes=("", "_post"))

for col in df_merged.columns:
    if any(k in col.lower() for k in ["kateg", "cat", "tema", "topik", "platform"]):
        print(f"\nSebaran '{col}' pada 1.000 komentar v1:")
        print(df_merged[col].value_counts(dropna=False).to_string())

if os.path.exists("docs/kategori_konten_v01.1.md"):
    print("\n=== ISI docs/kategori_konten_v01.1.md ===")
    with open("docs/kategori_konten_v01.1.md", "r", encoding="utf-8") as f:
        print(f.read())

<>:17: SyntaxWarning: invalid escape sequence '\k'
<>:17: SyntaxWarning: invalid escape sequence '\k'
/tmp/ipykernel_1049/3353047158.py:17: SyntaxWarning: invalid escape sequence '\k'
  - **Koefisien *Cohen's Kappa* ($\kappa$):** **0,5390** (kategori *Moderate Agreement*)


Bagian 4 docs/pedoman_label_final.md berhasil diperbarui dengan angka aktual D16!

Kolom posts_v0.csv: ['post_id', 'platform', 'akun', 'URL', 'tanggal', 'ringkasan_konteks', 'format', 'metrik_engagement', 'waktu_pencatatan', 'url', 'published_at']

Sebaran 'platform' pada 1.000 komentar v1:
platform
Instagram    1000

Sebaran 'waktu_pencatatan' pada 1.000 komentar v1:
waktu_pencatatan
NaN    1000

=== ISI docs/kategori_konten_v01.1.md ===
# Kategori Topik Konten Unggahan (Versi 0.1)

Berdasarkan _sampling_ unggahan pada tahap _pilot_, seluruh unggahan media sosial Pemprov Jawa Timur akan diklasifikasikan ke dalam 5 kategori utama dan 1 kategori cadangan berikut:

1.  **Layanan Publik & Administrasi**
    Mencakup informasi pelayanan masyarakat secara langsung, seperti jadwal Samsat, pembebasan pajak daerah, dan perizinan.
2.  **Kesejahteraan Sosial & Kebencanaan**
    Mencakup penanganan masalah warga seperti pendistribusian bantuan air bersih saat kekeringan, program pengasuhan keluar

**Menyusun Tiga Lembar Peluang Penelitian dalam format .docx (docs/peluang_konten.docx, docs/peluang_citra.docx, docs/peluang_layanan.docx).**

In [14]:
import os
import pandas as pd

try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
except ImportError:
    os.system("pip install -q python-docx")
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn

os.chdir("/content/drive/MyDrive/magang-ba-2026")
os.makedirs("docs", exist_ok=True)

# 1. Hitung angka aktual dari comments_v1.csv dan posts_v0.csv untuk mengisi bagian "Data tersedia"
df_comm = pd.read_csv("data/processed/comments_v1.csv")
df_posts = pd.read_csv("data/processed/posts_v0.csv").drop_duplicates(subset=["post_id"])
df_m = df_comm.merge(df_posts[["post_id", "platform", "format", "published_at"]], on="post_id", how="left")

total_comm = len(df_m)                 # 1000 baris komentar
unique_comm = df_m["comment_id"].nunique()  # 915 ID unik
total_posts = df_m["post_id"].nunique()

# Hitung jumlah komentar bertema layanan publik (Samsat, pajak, pemutihan, perizinan, pelayanan, desil, dll.)
kw_layanan = r"samsat|pajak|pemutihan|layanan|izin|perizinan|pkb|bbnkb|desil|bansos|bantuan|pelayanan|stnk"
n_layanan = int(
    df_m["context_text"].fillna("").str.lower().str.contains(kw_layanan).sum()
)

# Hitung jumlah komentar terkait citra / apresiasi / pimpinan / pemerintahan / prestasi
kw_citra = r"gubernur|pemprov|prestasi|kebanggaan|dirgahayu|hari jadi|rakor|forkopimda|penghargaan|pemerintahan"
n_citra = int(
    df_m["context_text"].fillna("").str.lower().str.contains(kw_citra).sum()
)

print(f"Angka Aktual Terhitung -> Total komentar: {total_comm} ({unique_comm} unik) | Unggahan induk: {total_posts} | Terkait Layanan: {n_layanan} | Terkait Citra/Pemerintahan: {n_citra}")

# Fungsi pembantu styling tabel & sel agar rapi dan muat 1 halaman A4
def set_cell_shading(cell, fill_hex):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), fill_hex)
    tcPr.append(shd)

def set_cell_margins(cell, top=80, bottom=80, left=130, right=130):
    tcPr = cell._tc.get_or_add_tcPr()
    tcMar = OxmlElement("w:tcMar")
    for m, val in [("top", top), ("bottom", bottom), ("left", left), ("right", right)]:
        node = OxmlElement(f"w:{m}")
        node.set(qn("w:w"), str(val))
        node.set(qn("w:type"), "dxa")
        tcMar.append(node)
    tcPr.append(tcMar)

def build_opportunity_docx(filepath, title_text, subtitle_text, rows_data):
    doc = Document()

    # Ukuran kertas A4 & margin ringkas agar pas 1 halaman
    sec = doc.sections[0]
    sec.page_width = Inches(8.27)
    sec.page_height = Inches(11.69)
    sec.top_margin = Inches(0.65)
    sec.bottom_margin = Inches(0.60)
    sec.left_margin = Inches(0.75)
    sec.right_margin = Inches(0.75)

    # Gaya font baku
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(10)
    style.font.color.rgb = RGBColor(0x1F, 0x29, 0x37)

    # Header Dokumen
    p_tag = doc.add_paragraph()
    p_tag.paragraph_format.space_before = Pt(0)
    p_tag.paragraph_format.space_after = Pt(2)
    r_tag = p_tag.add_run("LEMBAR PELUANG PENELITIAN · LABORATORIUM BUSINESS ANALYTICS (POLINEMA)")
    r_tag.font.size = Pt(8.5)
    r_tag.font.bold = True
    r_tag.font.color.rgb = RGBColor(0x1E, 0x40, 0xAF)

    p_title = doc.add_paragraph()
    p_title.paragraph_format.space_before = Pt(0)
    p_title.paragraph_format.space_after = Pt(3)
    r_title = p_title.add_run(title_text)
    r_title.font.size = Pt(14)
    r_title.font.bold = True
    r_title.font.color.rgb = RGBColor(0x11, 0x18, 0x27)

    p_sub = doc.add_paragraph()
    p_sub.paragraph_format.space_before = Pt(0)
    p_sub.paragraph_format.space_after = Pt(8)
    r_sub = p_sub.add_run(subtitle_text)
    r_sub.font.size = Pt(9.5)
    r_sub.font.italic = True
    r_sub.font.color.rgb = RGBColor(0x4B, 0x55, 0x63)

    # Tabel terstruktur 2 kolom sesuai contoh singkat Hari 38
    table = doc.add_table(rows=len(rows_data), cols=2)
    table.style = "Table Grid"
    table.autofit = False

    col_widths = [Inches(1.85), Inches(4.92)]
    for idx, (label, bullets) in enumerate(rows_data):
        row = table.rows[idx]
        c0, c1 = row.cells[0], row.cells[1]
        c0.width, c1.width = col_widths[0], col_widths[1]
        set_cell_shading(c0, "EEF2FF")
        set_cell_margins(c0, top=85, bottom=85, left=120, right=120)
        set_cell_margins(c1, top=85, bottom=85, left=130, right=130)

        p0 = c0.paragraphs[0]
        p0.paragraph_format.space_before = Pt(0)
        p0.paragraph_format.space_after = Pt(0)
        r0 = p0.add_run(label)
        r0.font.bold = True
        r0.font.size = Pt(9.8)
        r0.font.color.rgb = RGBColor(0x1E, 0x3A, 0x8A)

        for b_idx, b_text in enumerate(bullets):
            p1 = c1.paragraphs[0] if b_idx == 0 else c1.add_paragraph()
            p1.paragraph_format.space_before = Pt(0)
            p1.paragraph_format.space_after = Pt(2.5 if b_idx < len(bullets) - 1 else 0)
            p1.paragraph_format.line_spacing = 1.12
            r1 = p1.add_run(f"•  {b_text}")
            r1.font.size = Pt(9.5)

    # Catatan kaki tata kelola akademik (sesuai Langkah 5)
    p_foot = doc.add_paragraph()
    p_foot.paragraph_format.space_before = Pt(7)
    p_foot.paragraph_format.space_after = Pt(0)
    r_foot = p_foot.add_run(
        "Catatan: Dokumen ini merupakan inventaris peluang eksplorasi data awal untuk diskusi bersama pembimbing dan tim peneliti, "
        "bukan klaim temuan penelitian maupun rancangan publikasi."
    )
    r_foot.font.size = Pt(8.2)
    r_foot.font.italic = True
    r_foot.font.color.rgb = RGBColor(0x6B, 0x72, 0x80)

    doc.save(filepath)
    print(f"Berhasil membuat: {filepath} ({os.path.getsize(filepath)} bytes)")

Angka Aktual Terhitung -> Total komentar: 1000 (915 unik) | Unggahan induk: 49 | Terkait Layanan: 50 | Terkait Citra/Pemerintahan: 200


In [15]:
# ==============================================================================
# LEMBAR 1: docs/peluang_konten.docx
# ==============================================================================
rows_konten = [
    ("Judul lembar", [
        "Peluang penelitian: analisis konten dan pola respons komentar pada media sosial resmi Pemprov Jawa Timur"
    ]),
    ("Pertanyaan potensial", [
        "Bagaimana perbedaan distribusi sentimen komentar masyarakat antar-tema konten unggahan (Layanan Publik, Kesejahteraan Sosial, Ekonomi Daerah, Pemerintahan, serta Pendidikan & Lingkungan)?",
        "Apakah format penyajian konten (foto tunggal, karosel infografis, atau video/reels) berkaitan dengan tingkat pertanyaan prosedural maupun komentar emotikon singkat?",
        "Bagaimana pola perubahan volume komentar mingguan mengikuti momentum pengumuman kebijakan atau program daerah?"
    ]),
    ("Data tersedia", [
        f"Total {total_comm} baris komentar ({unique_comm} ID komentar unik) dari {total_posts} unggahan induk pada dataset v1 (comments_v1.csv & posts_v0.csv).",
        "Atribut lengkap: teks komentar teranonimisasi (text_original), ringkasan konteks unggahan (context_text), format konten (content_format), platform (Instagram), tanggal unggahan/komentar (rentang 1 Juli – 30 September 2026).",
        "Label sentimen hasil anotasi (802 Netral, 146 Positif, 52 Negatif), hasil adjudikasi (967 baris valid), dan skor prediksi IndoBERT (T3_indobert_len64, softmax) beserta 6 taksonomi kategori topik (kategori_konten_v01.1.md)."
    ]),
    ("Data yang masih kurang", [
        "Pelabelan kategori topik unggahan (6 kelas) belum terisi secara otomatis pada seluruh baris database SQLite sehingga masih memerlukan pemetaan penuh dari ringkasan konteks.",
        "Data lintas platform (khususnya komentar dari TikTok dan YouTube pada periode yang sama) masih terbatas dibandingkan dominasi sampel Instagram.",
        "Metrik jangkauan organik (impressions/reach) dan rincian jam tayang unggahan tidak tersedia pada data publik."
    ]),
    ("Risiko / keterbatasan", [
        "Sampel berasal dari unggahan terpilih pada akun resmi selama 3 bulan (Juli–September 2026) sehingga tidak merepresentasikan seluruh percakapan publik Jawa Timur.",
        "Dominasi kelas Netral (80,2%) dan komentar emotikon tunggal (102 baris) dapat memengaruhi sensitivitas analisis perbandingan sentimen antar-tema.",
        "Batas panjang token model (max_length = 64) berpotensi memotong konteks pada komentar aduan yang sangat panjang."
    ]),
    ("Langkah berikutnya yang diusulkan", [
        "Melengkapi pengisian kolom content_category pada tabel comments untuk seluruh unggahan induk berdasarkan pedoman kategori_konten_v01.1.md.",
        "Menyusun tabulasi silang antara kategori tema unggahan, format media, dan distribusi label sentimen terverifikasi.",
        "Mendiskusikan ruang lingkup analisis tematik lanjutan bersama dosen pembimbing dan tim peneliti Laboratorium Business Analytics."
    ])
]

In [16]:
# ==============================================================================
# LEMBAR 2: docs/peluang_citra.docx
# ==============================================================================
rows_citra = [
    ("Judul lembar", [
        "Peluang penelitian: dinamika citra institusi dan komunikasi publik dalam komentar media sosial"
    ]),
    ("Pertanyaan potensial", [
        "Bagaimana kecenderungan respons sentimen masyarakat pada unggahan yang menampilkan capaian prestasi daerah, peringatan hari jadi, dan agenda pimpinan daerah?",
        "Sejauh mana ungkapan sarkasme atau sindiran (kode SULIT-SARKASME) muncul pada unggahan bertema penghargaan atau seremoni pemerintahan?",
        "Bagaimana perbandingan proporsi komentar apresiatif (Positif) versus kritik langsung (Negatif) pada topik tata kelola pemerintahan dan keamanan wilayah?"
    ]),
    ("Data tersedia", [
        f"Tersedia {total_comm} komentar teranotasi ({unique_comm} ID unik), dengan sekitar {n_citra} komentar berada pada unggahan bertema pemerintahan, prestasi daerah, peringatan hari jadi, dan agenda pimpinan.",
        "Atribut tersedia: teks komentar asli (text_original), ringkasan konteks unggahan (context_text), catatan kasus khusus (termasuk penanda sarkasme/candaan dan protes keras), platform, dan tanggal publikasi (1 Juli – 30 September 2026).",
        "Tersedia label sentimen acuan (146 Positif, 802 Netral, 52 Negatif), nilai kesepakatan antar-anotator (Kappa = 0,5390 pada n = 230), serta fitur audit koreksi label (verification_history)."
    ]),
    ("Data yang masih kurang", [
        "Belum tersedia pengodean khusus dimensi citra institusi (misalnya: kredibilitas, empati komunikasi, transparansi, atau ketokohan pimpinan).",
        "Data pemberitaan media massa daring maupun percakapan di luar kanal resmi pemerintah (unsolicited mentions) belum tercakup sebagai pembanding.",
        "Informasi demografi atau latar belakang wilayah asal pengomentar tidak tersedia karena pembatasan privasi (anonimisasi PII)."
    ]),
    ("Risiko / keterbatasan", [
        "Komentar pada akun resmi pemerintah cenderung dipengaruhi oleh pengikut aktif atau akun penggerak sehingga tidak mencerminkan persepsi seluruh warga Jawa Timur.",
        "Sarkasme halus dan bahasa daerah dialek Jawa/Madura memiliki tingkat ambiguitas tinggi apabila dianalisis tanpa konteks berita yang sedang berkembang.",
        "Hasil analisis sentimen media sosial tidak dapat ditafsirkan sebagai indeks kepercayaan publik secara formal."
    ]),
    ("Langkah berikutnya yang diusulkan", [
        "Melakukan penyaringan subset komentar pada unggahan bertema pemerintahan dan prestasi daerah melalui menu Ekspor aplikasi.",
        "Meninjau ulang kasus berlabel sarkasme dan komentar negatif berkeyakinan tinggi bersama pembimbing untuk menyusun kerangka pengodean dimensi citra.",
        "Menyerahkan paket data terverifikasi kepada tim peneliti untuk penentuan arah kajian komunikasi publik."
    ])
]

In [17]:
# ==============================================================================
# LEMBAR 3: docs/peluang_layanan.docx
# ==============================================================================
rows_layanan = [
    ("Judul lembar", [
        "Peluang penelitian: persepsi layanan publik dalam komentar media sosial"
    ]),
    ("Pertanyaan potensial", [
        "Jenis informasi layanan publik apa yang paling sering memicu pertanyaan prosedural (kode SULIT-TANYA) maupun keluhan warga di kolom komentar?",
        "Bagaimana sebaran sentimen komentar pada unggahan terkait layanan administrasi (seperti Samsat, pembebasan/pemutihan pajak daerah, dan bantuan sosial)?",
        "Aspek hambatan layanan apa yang paling banyak muncul dalam komentar bernada Negatif dan laporan pengaduan masyarakat?"
    ]),
    ("Data tersedia", [
        f"Tersedia {total_comm} komentar pada dataset v1, dengan {n_layanan} komentar teridentifikasi pada konteks unggahan terkait layanan publik, pajak daerah/Samsat, perizinan, dan bantuan sosial.",
        "Atribut tersedia: teks komentar (text_original), target frasa konteks (context_target_phrase), catatan anotasi (permintaan informasi, keluhan teknis, masukan/saran), label sentimen, skor prediksi IndoBERT, platform, dan tanggal (1 Juli – 30 September 2026).",
        "Tersedia riwayat koreksi manual (verification_history) untuk menelusuri alasan perubahan label pada komentar pengaduan layanan."
    ]),
    ("Data yang masih kurang", [
        "Sub-kategori jenis layanan spesifik (misalnya pajak kendaraan, administrasi kependudukan, pendidikan, kesehatan, infrastruktur) belum dikodekan per baris komentar.",
        "Data pembanding di luar media sosial—seperti log pengaduan SP4N-LAPOR!, rekapitulasi pusat layanan (call center), atau hasil Survei Kepuasan Masyarakat (SKM)—belum tersedia.",
        "Status tindak lanjut atau penyelesaian dari admin instansi terhadap pertanyaan warga di kolom balasan belum direkam."
    ]),
    ("Risiko / keterbatasan", [
        "Sampel hanya mencakup komentar pada unggahan media sosial resmi periode Juli–September 2026 sehingga tidak mewakili seluruh pengguna layanan publik di Jawa Timur.",
        "Proporsi komentar netral berbentuk pertanyaan administratif sangat tinggi (kode SULIT-TANYA), sehingga distribusi sentimen tidak dapat dijadikan ukuran indeks kepuasan layanan.",
        "Keluhan warga sering ditulis dalam bentuk balasan berantai yang memerlukan konteks komentar sebelumnya."
    ]),
    ("Langkah berikutnya yang diusulkan", [
        "Menambahkan taksonomi sub-kategori jenis layanan publik pada subset komentar berkonteks layanan bersama tim peneliti.",
        "Mengekstrak daftar pertanyaan layanan berulang sebagai bahan evaluasi alur informasi FAQ bagi pengelola kanal komunikasi.",
        "Mengkonsultasikan potensi integrasi data komentar media sosial dengan data pengaduan resmi bersama dosen pembimbing."
    ])
]

build_opportunity_docx(
    "docs/peluang_konten.docx",
    "Peluang Penelitian: Analisis Konten dan Pola Respons Publik",
    "Eksplorasi tematik unggahan dan sebaran sentimen komentar pada kanal media sosial resmi Pemprov Jawa Timur",
    rows_konten,
)

build_opportunity_docx(
    "docs/peluang_citra.docx",
    "Peluang Penelitian: Citra Institusi dan Komunikasi Publik",
    "Kajian dinamika apresiasi, kritik, dan sarkasme masyarakat pada unggahan tata kelola pemerintahan dan prestasi daerah",
    rows_citra,
)

build_opportunity_docx(
    "docs/peluang_layanan.docx",
    "Peluang Penelitian: Persepsi Layanan Publik dalam Komentar",
    "Pemetaan pertanyaan prosedural, masukan teknis, dan keluhan warga terhadap informasi layanan publik di media sosial",
    rows_layanan,
)


Berhasil membuat: docs/peluang_konten.docx (38636 bytes)
Berhasil membuat: docs/peluang_citra.docx (38582 bytes)
Berhasil membuat: docs/peluang_layanan.docx (38521 bytes)


In [18]:
# ==============================================================================
# VERIFIKASI AKHIR SELURUH 7 BERKAS LUARAN HARI 38
# ==============================================================================
print("\n=== REKAP VERIFIKASI SELURUH BUKTI & CHECKLIST HARI 38 ===")
deliverables = [
    "README.md",
    "docs/data_dictionary_final.md",
    "docs/pedoman_label_final.md",
    "docs/petunjuk_aplikasi_final.md",
    "docs/peluang_konten.docx",
    "docs/peluang_citra.docx",
    "docs/peluang_layanan.docx",
]
for fp in deliverables:
    ok = os.path.exists(fp) and os.path.getsize(fp) > 0
    print(f" [{'OK' if ok else 'GAGAL'}] {fp} ({os.path.getsize(fp) if ok else 0} bytes)")


=== REKAP VERIFIKASI SELURUH BUKTI & CHECKLIST HARI 38 ===
 [OK] README.md (4177 bytes)
 [OK] docs/data_dictionary_final.md (8803 bytes)
 [OK] docs/pedoman_label_final.md (5168 bytes)
 [OK] docs/petunjuk_aplikasi_final.md (3918 bytes)
 [OK] docs/peluang_konten.docx (38636 bytes)
 [OK] docs/peluang_citra.docx (38582 bytes)
 [OK] docs/peluang_layanan.docx (38521 bytes)
